# Ingredient Decomposition

## 01: RxNorm

In [1]:
from VaxMapper.src.rxnorm_term_getter import parallel_map, gather_vaccine_related_rxcuis, get_obsolete_rxcuis, get_rxnorm_status,decompose_concepts, get_fda_spl_id
import pandas as pd

In [ ]:
rx_vax_ids= gather_vaccine_related_rxcuis()
obs_rx_ids = get_obsolete_rxcuis()

In [ ]:
rx_vax_df = pd.DataFrame(parallel_map(get_rxnorm_status,list(rx_vax_ids)))
obs_rx_df = pd.DataFrame(obs_rx_ids, columns=['rxcui','name','tty'])
obs_rx_df['status'] = 'Obsolete'

In [13]:
rx_vax_df = pd.read_pickle('mresults/rxnav_df.pkl')

In [14]:
rx_vax_df

,rxcui,name,status,tty
0,2566415,Streptococcus pneumoniae serotype 1 capsular a...,Active,SCDF
1,1153476,adenovirus type 7 vaccine live Oral Product,Active,SCDG
2,2624741,Zaire ebolavirus (strain Kikwit-95) envelope g...,Active,PIN
3,1601404,meningococcal group B vaccine 0.1 MG/ML / Neis...,Active,SBDC
4,2587056,Bordetella pertussis filamentous hemagglutinin...,Active,SBDC
...,...,...,...,...
3084,1156379,"mumps virus vaccine live, Jeryl Lynn strain / ...",Obsolete,SCDG
3085,2563025,Fluzone Quadrivalent 2021-2022 Injectable Product,Obsolete,SBDG
3086,2054184,influenza A virus (H1N1) antigen / influenza A...,Obsolete,SBDF
3087,1664640,influenza A virus A/Bolivia/559/2013 (H1N1) an...,Obsolete,SBDC


In [4]:
# select all from rx_vax_df where tty is not in IN or PIN
tty_list = ('IN', 'PIN', 'MIN')
non_in_pin_df = rx_vax_df[~rx_vax_df['tty'].isin(tty_list)]
non_in_pin_df

,rxcui,name,status,tty
0,2566415,Streptococcus pneumoniae serotype 1 capsular a...,Active,SCDF
1,1153476,adenovirus type 7 vaccine live Oral Product,Active,SCDG
3,1601404,meningococcal group B vaccine 0.1 MG/ML / Neis...,Active,SBDC
4,2587056,Bordetella pertussis filamentous hemagglutinin...,Active,SBDC
5,1177669,RabAvert Injectable Product,Active,SBDG
...,...,...,...,...
3084,1156379,"mumps virus vaccine live, Jeryl Lynn strain / ...",Obsolete,SCDG
3085,2563025,Fluzone Quadrivalent 2021-2022 Injectable Product,Obsolete,SBDG
3086,2054184,influenza A virus (H1N1) antigen / influenza A...,Obsolete,SBDF
3087,1664640,influenza A virus A/Bolivia/559/2013 (H1N1) an...,Obsolete,SBDC


In [9]:
tty_list = ('IN', 'PIN')

In [6]:
rxcui = non_in_pin_df['rxcui'].to_list()
decomp = []
for rx in rxcui:
    parts = decompose_concepts(rx, tuple(tty_list))
    for part in parts:
        decomp.append([part,rx])

decomp_df = pd.DataFrame(decomp, columns=['part','source_rxcui'])

In [8]:
decomp_df['in_rxcui'] = decomp_df['part'].apply(lambda x: x['rxcui'])
decomp_df['in_name'] = decomp_df['part'].apply(lambda x: x['name'])
decomp_df['in_tty'] = decomp_df['part'].apply(lambda x: x['tty'])
decomp_df

,part,source_rxcui,in_rxcui,in_name,in_tty
0,"{'rxcui': '2566296', 'name': 'Streptococcus pn...",2566415,2566296,Streptococcus pneumoniae serotype 33F capsular...,IN
1,"{'rxcui': '2566297', 'name': 'Streptococcus pn...",2566415,2566297,Streptococcus pneumoniae serotype 22F capsular...,IN
2,"{'rxcui': '798220', 'name': 'Streptococcus pne...",2566415,798220,Streptococcus pneumoniae serotype 14 capsular ...,IN
3,"{'rxcui': '798222', 'name': 'Streptococcus pne...",2566415,798222,Streptococcus pneumoniae serotype 18C capsular...,IN
4,"{'rxcui': '798224', 'name': 'Streptococcus pne...",2566415,798224,Streptococcus pneumoniae serotype 19F capsular...,IN
...,...,...,...,...,...
9537,"{'rxcui': '1008803', 'name': 'Streptococcus pn...",901622,1008803,Streptococcus pneumoniae serotype 1 capsular a...,MIN
9538,"{'rxcui': '2566305', 'name': 'Streptococcus pn...",901622,2566305,Streptococcus pneumoniae serotype 1 capsular a...,MIN
9539,"{'rxcui': '2588217', 'name': 'hepatitis B surf...",2645465,2588217,"hepatitis B surface antigen (isoform L), recom...",PIN
9540,"{'rxcui': '2588218', 'name': 'hepatitis B surf...",2645465,2588218,"hepatitis B surface antigen (isoform M), recom...",PIN


In [9]:
decomp_df['in_rxcui'].nunique()

270

In [10]:
decomp_df.to_pickle('mresults/rxnorm_in_pin_decomp.pkl')

## 02. FDA SPL

#### Extracting SPL ID's for RxCuis

In [2]:
from VaxMapper.src.rxnorm_term_getter import parallel_map

In [3]:
rx_vax_df = pd.read_pickle('mresults/rxnav_df.pkl')

In [4]:
rxcui = rx_vax_df['rxcui'].to_list()
spl_id_results = parallel_map(get_fda_spl_id, rxcui)

: 

: 

In [ ]:
spl_id_results

[{'rxcui': '2624741',
  'propName': 'SPL_SET_ID',
  'propValue': '2b1db7a1-c3f2-42e5-8499-91724edbf65e'},
 {'rxcui': '798222',
  'propName': 'SPL_SET_ID',
  'propValue': '1158fa93-ef41-4a29-8252-9251f94c53c8'},
 {'rxcui': '854936',
  'propName': 'SPL_SET_ID',
  'propValue': '4c6db457-2514-48b9-b48a-fa6258aba116'},
 {'rxcui': '2686978',
  'propName': 'SPL_SET_ID',
  'propValue': '1a264c2f-d2f8-4ab5-bc5e-fbed0001ede6'},
 {'rxcui': '2641622',
  'propName': 'SPL_SET_ID',
  'propValue': '03f2c9fc-534b-49ec-9113-81938b1eadb9'},
 {'rxcui': '2719311',
  'propName': 'SPL_SET_ID',
  'propValue': '0c3cd915-babe-49c0-8aac-bf0247a17f3e'},
 {'rxcui': '2566422',
  'propName': 'SPL_SET_ID',
  'propValue': '1158fa93-ef41-4a29-8252-9251f94c53c8'},
 {'rxcui': '1597103',
  'propName': 'SPL_SET_ID',
  'propValue': 'a21f4f4b-b891-4f25-b747-cb9ec7d865d6'},
 {'rxcui': '797635',
  'propName': 'SPL_SET_ID',
  'propValue': '4d8781ff-9366-462c-8161-6e958f44fcb4'},
 {'rxcui': '2360572',
  'propName': 'SPL_SET_ID',

In [6]:
spl_df = pd.DataFrame(spl_id_results, columns=['rxcui','propName','propValue'])
spl_df

NameError: name 'spl_id_results' is not defined

In [18]:
spl_df.to_pickle('mresults/rxnav_spl_df.pkl')

In [4]:
spl_df = pd.read_pickle('mresults/rxnav_spl_df.pkl')
spl_df

,rxcui,propName,propValue
0,2624741,SPL_SET_ID,2b1db7a1-c3f2-42e5-8499-91724edbf65e
1,798222,SPL_SET_ID,1158fa93-ef41-4a29-8252-9251f94c53c8
2,854936,SPL_SET_ID,4c6db457-2514-48b9-b48a-fa6258aba116
3,2686978,SPL_SET_ID,1a264c2f-d2f8-4ab5-bc5e-fbed0001ede6
4,2641622,SPL_SET_ID,03f2c9fc-534b-49ec-9113-81938b1eadb9
...,...,...,...
323,805573,SPL_SET_ID,f3182470-1965-4e20-dbaf-e3506f893ea5
324,1994352,SPL_SET_ID,d61c7f99-438b-4cac-ae10-997ff4c0fd1f
325,2719212,SPL_SET_ID,5fe0ce14-bc11-4b43-b002-7efdca0d3003
326,2274413,SPL_SET_ID,89191ae9-7f2b-4206-8397-bf7fce3436ac


In [5]:
import requests
from lxml import etree

DAILYMED_BASE = "https://dailymed.nlm.nih.gov/dailymed/services/v2"
CONTRA_Loinc = "34070-3"
ADVERSE_Loinc = "34084-4"
# https://dailymed.nlm.nih.gov/dailymed/services/v2/spls/{SETID}.xml

In [16]:
spl_df['propValue'].nunique()

126

In [8]:
class DailyMedError(Exception):
    pass

def fetch_spl_xml_by_setid(setid: str, timeout: int = 30) -> bytes:
    """
    Download the SPL XML for a given DailyMed SETID.
    """
    url = f"{DAILYMED_BASE}/spls/{setid}.xml"  # returns the full SPL document (v2)
    resp = requests.get(url, timeout=timeout)
    if resp.status_code != 200 or not resp.content.strip():
        raise DailyMedError(f"Failed to fetch SPL XML for setid={setid}: {resp.status_code}")
    return resp.content

def parse_xml(xml_bytes: bytes) -> etree._Element:
    """
    Parse bytes into an lxml Element tree with recovery on.
    """
    parser = etree.XMLParser(remove_blank_text=True, recover=True, huge_tree=True)
    return etree.fromstring(xml_bytes, parser=parser)

def get_default_ns(root: etree._Element) -> str:
    """
    SPLs typically use the default namespace urn:hl7-org:v3. Detect it from the root.
    """
    return root.nsmap.get(None, "urn:hl7-org:v3")

def find_section_by_loinc(root: etree._Element, loinc_code: str) -> etree._Element | None:
    """
    Return the <section> element whose <code/@code> == loinc_code.
    Works regardless of the section @ID (e.g., s10).
    """
    ns = {"hl7": get_default_ns(root)}
    # Find the section by its LOINC code anywhere in the document
    xpath = ".//hl7:component[hl7:section/hl7:code[@code=$code]]/hl7:section"
    results = root.xpath(xpath, namespaces=ns, code=loinc_code)
    return results[0] if results else None

def section_text(section_el: etree._Element) -> str:
    """
    Extract human-readable text from the section's <text> narrative block.
    Preserves paragraph/list breaks reasonably well.
    """
    if section_el is None:
        return ""
    ns = {"hl7": section_el.nsmap.get(None, "urn:hl7-org:v3")}
    text_el = section_el.find(".//hl7:text", namespaces=ns)
    if text_el is None:
        return ""
    # Convert the narrative (which may include <paragraph>, <list>, etc.) to plain text
    text = etree.tostring(text_el, method="text", encoding="unicode")
    # Normalize common non-breaking spaces and excess whitespace
    text = text.replace("\xa0", " ").strip()
    # Optional: collapse sequences of blank lines
    lines = [ln.strip() for ln in text.splitlines()]
    lines = [ln for ln in lines if ln]  # drop empty lines
    return "\n".join(lines)

# def extract_contraindications_section(setid: str) -> dict:
#     """
#     Fetch SPL by setid and extract the Contraindications section.
#     Returns:
#         {
#           "setid": str,
#           "found": bool,
#           "section_xml": str | None,
#           "section_text": str | None
#         }
#     """
#     xml_bytes = fetch_spl_xml_by_setid(setid)
#     root = parse_xml(xml_bytes)

#     section = find_section_by_loinc(root, CONTRA_Loinc)
#     if section is None:
#         return {"setid": setid, "found": False, "section_xml": None, "section_text": None}

#     # Pretty XML string for the section only (optional, handy for debugging)
#     section_xml = etree.tostring(section, pretty_print=True, encoding="unicode")

#     return {
#         "setid": setid,
#         "found": True,
#         "section_xml": section_xml,
#         "section_text": section_text(section)
#     }

def extract_section(setid: str, loinc_code: list[str]) -> dict:
    """
    Fetch SPL by setid and extract a specific section.
    Returns:
        {
          "setid": str,
          "found": bool,
          "section_xml": str | None,
          "section_text": str | None
        }
    """
    xml_bytes = fetch_spl_xml_by_setid(setid)
    root = parse_xml(xml_bytes)

    sections = {}
    for code in loinc_code:
        section = find_section_by_loinc(root, code)
        if section is not None:
            sections[code] = {
                "section_xml": etree.tostring(section, pretty_print=True, encoding="unicode"),
                "section_text": section_text(section)
            }

    return {
        "setid": setid,
        "found": bool(sections),
        "sections": sections
    }
    # return {
    #     "setid": setid,
    #     "found": True,
    #     "section_xml": section_xml,
    #     "section_text": section_text(section)
    # }

### Contraindication Extraction

In [9]:
result = extract_contraindications_section("0146011c-944b-4251-8a5f-e573a3d5f8c0")

NameError: name 'extract_contraindications_section' is not defined

In [11]:
result['sections'].keys()

dict_keys(['34070-3', '34084-4'])

In [28]:
fda_label_spl_list = ['c7d0b43c-7250-4809-8a05-fcde5862f076',
'e0b11800-8922-11df-b3d7-0002a5d5c51b',
'd4f29180-7c76-40ac-9341-cf62702c4090',
'f136b41b-56f3-43a9-9d6a-61b2afca5b4a',
'a83f0b99-9038-4c5a-aaac-8792b32838fe',
'63d5cd82-02e3-4436-b81a-12d45bb6a90b',
'5e9a8ff0-3fdf-448a-97b3-83e6b904d864',
'2d33d9ae-1fe6-477e-9285-8890f6e98be4',
'1a87f53c-0c8a-4c66-9048-2a0cf60a6a5c',
'db0c7f7c-7624-477d-ab7c-310a903c7025',
'a41b7601-34f2-4a88-a406-f53011fb7de1',
'1d8930f5-434d-4bfa-befc-b54913c8dd36',
'ce88effe-7ff2-418f-acb3-3cb5d5d2bf95',
'06f34d0f-4e72-41d3-967f-8abf3f2005c1',
'97c27eb6-1d2e-4953-afac-882ada70c6f8',
'48c86164-de07-4041-b9dc-f2b5744714e5',
'f96b315c-fa57-4876-a7e5-a9b584d8e6e6',
'280ee057-3488-4d77-b510-8bc733eeca1e',
'f8b2634b-ccd7-4a42-8db3-26ba9e9627e6',
'6b5f89e9-1292-4f13-5590-44c874bf299c',
'76798bc5-6752-4c02-9f03-68a361eea66b',
'dd047022-e51c-4aa1-bd68-bc23161c3ce3',
'b63c4c7d-3dbf-419d-84cc-2c1957b92be7',
'e7a819d7-a597-4c91-9078-7cd20ac34410',
'67d5c665-ca88-49ba-b7d3-7145d9878cf0',
'de16dd6a-859b-4180-c6af-f930be14f26a',
'76504ed2-6417-4176-bc31-24eccb8b5584',
'3ed8472a-c6eb-4076-9d66-025ede589e3d',
'174faf80-eb4f-4f11-b875-d74bd26c2e65',
'9dff46c6-4b15-4d10-aca6-d5ef3735a530',
'745ff8df-1618-4b76-9aa1-6f42752c0dda',
'8143d01c-4911-40db-95b2-47f3ebea2a7d',
'18e8f9fc-4d62-4d30-4ab7-bac03336171e',
'dad367dd-7cb8-4344-9920-8710149473d2',
'f9499a4d-1288-4bd3-9d59-1d72092c38cd',
'7df68090-7e98-4b01-9c29-0b941a1c307d',
'618a7f1b-7162-4d16-9076-51b8aab4c8d2',
'135ff1b4-c21f-4df1-97be-9475680e9e44',
'63d283dd-e857-4c0e-b1fa-9e9d9b8717a8',
'89191ae9-7f2b-4206-8397-bf7fce3436ac',
'dcecf9e0-6c28-4964-9ddb-9379705fa26c',
'2ec65f7e-4aa2-4b41-b578-885ea59d6e9d',
'b5bd99e4-569d-48d5-ba75-16e69f8c409a',
'36e9a878-d6c2-4808-9915-294f47b1fe3d',
'e80cfadc-71b6-4881-b5f7-2be67adbf8b8',
'47ef5267-5624-48d4-91e2-483e93755f6f',
'f1ad4bca-839d-41cd-a132-a6984780912e',
'9a4037a2-6c63-457a-8d6f-2fd05d926ea2',
'618db7b8-a5e4-49c8-9390-38912e6f39fd',
'd61c7f99-438b-4cac-ae10-997ff4c0fd1f',
'a21f4f4b-b891-4f25-b747-cb9ec7d865d6',
'd099cf95-a7cc-4636-994d-226c3f866cdc',
'8355d971-d377-4d1d-973e-a65798bce7f9',
'30952400-0572-4431-9150-3a41affffb9a',
'ab802261-3eeb-4eea-af6a-31c830ac91f8',
'fc5cfd1c-1ff5-4cfa-b544-db8d7f26d46f',
'3513dfb0-4d62-4ad1-bb15-75c7555896ff',
'05822ec0-a82a-11e7-81fe-424c58303031',
'1adcb911-4e6b-7b6f-e063-6394a90a8264',
'29b78835-6a39-54ad-e063-6294a90a82cb',
'de0e2089-f781-4d47-99f5-c2a1b7b0f1eb',
'5fe0ce14-bc11-4b43-b002-7efdca0d3003',
'96ca1b4c-67bf-3db6-24c5-c4841b073c66',
'ced9a8b6-ef2c-a620-c61c-ff9374249543',
'03f2c9fc-534b-49ec-9113-81938b1eadb9',
'9dbbf304-7be3-4417-8285-a8f5fd20f977',
'd041603b-fa96-433c-a9bc-5393aab4a289',
'64f608b7-ad43-4395-b806-d4216175c8d3',
'8213c229-a67a-4d3f-bd8f-b8729ae28472',
'a06c4ead-c988-4b10-b832-4d7460fa358f',
'67de8652-2e5a-4d18-aee9-d9b789ccfe46',
'0c3cd915-babe-49c0-8aac-bf0247a17f3e',
'0d4af785-f5e8-4f6e-91f6-2ece6ab58d5c',
'b8f34cfb-ee00-4b4a-aff8-643fe6f1d9e1',
'dccfb747-8f64-46ec-3993-95195b585581',
'61d4995d-92be-4678-74b9-79b3e12e5e30',
'276b84fb-096b-3f91-e054-00144ff8d46c',
'e654303b-b7b1-4e5b-a0a1-e999110060bf',
'b68b7911-628d-4932-b0e8-4ae72c48f92f',
'95c6fdb6-b587-4413-92f9-d592b9f7a23e',
'252968ca-c714-4c1c-9e60-0b699cb9362f',
'0a9e384f-e717-436b-b9a0-15e53cef0862',
'73eae9fc-507b-4c9c-883d-63eb2e3cc6f6',
'6b2c9ee3-ae88-46f1-9b3f-fc9e9e716cbf',
'2dc730d5-55fd-4e98-8c8a-daa7d8f872b0',
'bcaf5f75-caaf-41fd-875b-5800310070d1',
'fd5652b6-5ae4-437e-a456-47deaf500794',
'4d8781ff-9366-462c-8161-6e958f44fcb4',
'ab1c3318-578e-4d0c-a21e-fc6df07e9fb7',
'6046583d-f1a5-4620-80bc-03992b6525d0',
'f70cf2fc-6e6d-4a74-9f7a-db8fec072fd7',
'1a20513c-d255-46d7-8e6b-294cec4b44f4',
'5d49181b-b974-a5da-3b38-12a3a87bb96b',
'e8c6b638-c22f-4c20-ba40-b9d5e85c9541',
'1158fa93-ef41-4a29-8252-9251f94c53c8',
'3a33a84f-166b-4698-90b9-d79265234ae7',
'd4e2cf51-e6a8-4103-bb1d-6120c6474ff8',
'1a264c2f-d2f8-4ab5-bc5e-fbed0001ede6',
'bb362a20-6d91-4ae8-bebb-9ee8b2591814',
'4c97c74d-b6b5-4492-b7ed-e02c76cc8514',
'4c6db457-2514-48b9-b48a-fa6258aba116',
'34a647f5-8728-451b-b918-94c8acd15974',
'6979f24b-5f49-4190-a4e6-c1523e0d3108',
'84b7a672-eeb1-4527-84ac-68196b156be2',
'fd2f21f0-9f8e-4abd-b603-e1834a252c2d',
'ba8d4e72-f452-4859-ae6f-3644b4b0a78c',
'be18292e-b1a2-4815-a0ed-003efaa6bea3',
'e5c837e7-41e8-496a-9c85-6b0453b35948',
'1e50f275-002e-413f-a840-66ee3cb3740c',
'f3182470-1965-4e20-dbaf-e3506f893ea5',
'aaf3b24e-85fd-43ee-b657-2ee4df312ec3',
'ad1fbe7f-2995-45dd-92f3-7baccaab85d9',
'5f674a82-00fb-41f8-99a5-df8985771324',
'21b75f1d-1cda-49e5-8176-cc23827824f3',
'09d800e3-427e-4973-9261-61592d662bbd',
'194fde77-eaf8-4e4c-84ef-35d67b5eec11',
'613aaac9-ec18-4b22-addb-599e1193e6f5',
'70bfb8c1-6d1a-4b0a-8a21-7a7ed8e748c9',
'c3c03458-942c-4565-8a9c-1901bb0d2db0',
'cd98bff9-4602-4268-d68d-029a14a5513b',
'46993726-584e-4907-ac4d-94e59b236b82',
'da102644-62a9-41c2-8a91-6fc19a03e6f1',
'69fc3841-a461-4706-bdcb-1176c40bf486',
'd3a52eba-83d2-4fd1-8db5-dd04bcb1ba9d',
'524cf052-e90e-4595-af0a-608edbe9bd31',
'2b1db7a1-c3f2-42e5-8499-91724edbf65e',
'0280849d-5c78-4a9d-8941-4eab429f6bd8',
'279a251c-f4d0-4c47-b999-7ef1a45ac0d0',
'e9ba0102-8cbb-4c4c-810e-292119095a8a']

In [34]:
# spl_id = spl_df['propValue'].unique().tolist()
rx_spl_df = pd.DataFrame(columns=['spl_id', 'contra_text', 'adverse_text'])
rx_spl_df['spl_id'] = fda_label_spl_list
def _safe_get_contra(sid):
    try:
        res = extract_section(sid, [CONTRA_Loinc, ADVERSE_Loinc])
        if res.get("found"):
            return res["sections"].get(CONTRA_Loinc, {}).get("section_text", "")
        return ""
    except Exception:
        return ""

def _safe_get_adverse(sid):
    try:
        res = extract_section(sid, [CONTRA_Loinc, ADVERSE_Loinc])
        if res.get("found"):
            return res["sections"].get(ADVERSE_Loinc, {}).get("section_text", "")
        return ""
    except Exception:
        return ""

rx_spl_df['contra_text'] = rx_spl_df['spl_id'].apply(_safe_get_contra)
rx_spl_df['adverse_text'] = rx_spl_df['spl_id'].apply(_safe_get_adverse)
# rx_spl_df['adverse_text'] = ''


In [4]:
rx_spl_df

NameError: name 'rx_spl_df' is not defined

In [36]:
rx_spl_df.to_csv('results/vaccine_spl_contra_adverse_sections.csv', index=False)

In [2]:
rx_spl_df = pd.read_csv('results/vaccine_spl_contra_adverse_sections.csv')
rx_spl_df

,spl_id,contra_text,adverse_text
0,c7d0b43c-7250-4809-8a05-fcde5862f076,"Pregnancy (4.1, 8.1).Severe allergic reaction ...",NaN
1,e0b11800-8922-11df-b3d7-0002a5d5c51b,Do not administer BioThrax to individuals with...,The most common (≥10%) local (injection-site) ...
2,d4f29180-7c76-40ac-9341-cf62702c4090,Do not administer CYFENDUS to individuals with...,The most common (≥10%) injection-site adverse ...
3,f136b41b-56f3-43a9-9d6a-61b2afca5b4a,NaN,NaN
4,a83f0b99-9038-4c5a-aaac-8792b32838fe,BCG VACCINE for prevention of tuberculosis sho...,Although BCG vaccination often causes local re...
...,...,...,...
124,524cf052-e90e-4595-af0a-608edbe9bd31,History of severe allergic reaction to any com...,Frequently reported (≥10%) adverse reactions i...
125,2b1db7a1-c3f2-42e5-8499-91724edbf65e,Do not administer ERVEBO to individuals with a...,The most commonly reported local and systemic ...
126,0280849d-5c78-4a9d-8941-4eab429f6bd8,Do not administer SHINGRIX to anyone with a hi...,•Solicited local adverse reactions reported in...
127,279a251c-f4d0-4c47-b999-7ef1a45ac0d0,Do not administer SHINGRIX to anyone with a hi...,•Solicited local adverse reactions reported in...


## Baseline Testing - Contraindications Identification

In [3]:
# from VaxMapper.src.llm import load_model_local
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0,1,2,3'

In [ ]:
# >>> from llm_runner import load_model_local, build_messages_from_iter, xml_section_iter
# >>> llm = load_model_local("meta-llama/Llama-3.1-8B-Instruct")
# >>> system = "You extract adverse reactions. Reply JSON only."
# >>> sections = xml_section_iter("TENIVAC.xml", tags=["ADVERSE REACTIONS", "CONTRAINDICATIONS"])
# >>> msgs = build_messages_from_iter(system, "Section: {section}\n\n{text}", sections)
# >>> out = [llm.generate(m, max_new_tokens=256) for m in msgs]
# >>> print(out[0]["text"])

In [4]:
from VaxMapper.src.llm import load_model_local, build_messages_from_iter

In [5]:
model = load_model_local("google/medgemma-27b-text-it")

Loading checkpoint shards:   0%|          | 0/11 [00:00<?, ?it/s]

The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


In [ ]:
## LARGEST / EXPLICIT
system = """
You are a biomedical NLP assistant that identifies CONTRAINDICATIONS in regulatory drug or vaccine documents.
Your ONLY task is to find text spans that express a contraindication and return them as JSON with character offsets.

--------------------
DEFINITIONS
--------------------
A contraindication is a condition or situation where the product SHOULD NOT be used or administered.

Typical patterns include:
- Explicit phrases:
  - "is contraindicated in patients with ..."
  - "contraindicated in pregnancy"
  - "do not use in patients with ..."
  - "should not be administered to individuals with ..."
  - "must not be given to patients who ..."
- Allergy/hypersensitivity:
  - "history of severe allergic reaction to any component of the vaccine"
  - "known hypersensitivity to neomycin"
- Specific clinical conditions:
  - "immunocompromised patients"
  - "severe immunodeficiency"
  - "patients with a history of anaphylaxis"

NOT contraindications:
- General warnings, precautions, and risk factors unless they explicitly say the product should not be used.
- Dosing instructions, monitoring advice, contact information, or storage instructions.
- Statements that say there are NO contraindications (e.g., "This product has no known contraindications").

--------------------
HANDLING NEGATION & SCOPE
--------------------
- Ignore negated statements such as:
  - "There are no known contraindications."
  - "Use is not contraindicated in patients with ..."
- Only mark spans where the text clearly indicates that use IS prohibited, discouraged, or "contraindicated".

--------------------
OUTPUT FORMAT
--------------------
You MUST return ONLY a single JSON object with the following structure:

{
  "items": [
    {
      "start": <int>,          // character offset of the FIRST character of the contraindication span in the provided TEXT
      "len": <int>,            // length in characters of the span
      "span_text": "<verbatim contraindication span>",
      "condition_text": "<short phrase naming the condition or situation>",
      "type": "<one of: 'clinical_condition' | 'allergy_or_hypersensitivity' | 'other'>"
    },
    ...
  ]
}

Rules:
- "start" and "len" are relative to the EXACT TEXT the user provides (starting at index 0).
- "span_text" must be copied verbatim from the input text for that span.
- "condition_text" should be a concise, normalized description of the contraindicated condition (e.g., "severe immunodeficiency", "pregnancy", "history of anaphylaxis to neomycin").
- Use "allergy_or_hypersensitivity" when the text describes an allergy, hypersensitivity, or severe allergic reaction to a substance or component.
- Use "clinical_condition" for diseases, physiological states, or patient characteristics (e.g., pregnancy, immunodeficiency, cardiac failure).
- Use "other" only when it does not clearly fit the above categories but is still a contraindication.
- Do NOT include overlapping or duplicate spans; choose the minimal span that clearly expresses the contraindication.
- If there are no contraindications in the text, return: {"items": []}

--------------------
TASK
--------------------
The user will provide a block of TEXT (e.g., the CONTRAINDICATIONS section of an SPL).

Carefully read the TEXT and output ONLY the JSON object described above.
Do not include any explanations, comments, or additional text outside the JSON.
"""

In [ ]:
## MINIMAL
system = """
You are a biomedical NLP assistant that identifies CONTRAINDICATIONS in regulatory drug or vaccine documents.
Your ONLY task is to find text spans that express a contraindication and return them as JSON with character offsets.

--------------------
DEFINITIONS
--------------------
A contraindication is a condition or situation where the product SHOULD NOT be used or administered.

--------------------
OUTPUT FORMAT
--------------------
You MUST return ONLY a JSON object with the following structure:

{{
  "items": [
    {{
      "span_text": "<verbatim contraindication span>",
      "condition_text": "<short phrase naming the condition or situation>"
    }},
    ...
  ]
}}

Rules:
- "start" and "len" are relative to the EXACT TEXT the user provides (starting at index 0).
- "span_text" must be copied verbatim from the input text for that span.
- "condition_text" should be a concise, normalized description of the contraindicated condition (e.g., "severe immunodeficiency", "pregnancy", "history of anaphylaxis to neomycin").
- Do NOT include overlapping or duplicate spans; choose the minimal span that clearly expresses the contraindication.
- If there are no contraindications in the text, return: {{"items": []}}

--------------------
TASK
--------------------
The user will provide a block of TEXT (e.g., the CONTRAINDICATIONS section of an SPL).

Carefully read the TEXT and output ONLY the JSON object described above.
Do not include any explanations, comments, or additional text outside the JSON. Do not show your reasoning. Provide only the final answer.
"""

user = """
Here is the CONTRAINDICATIONS section from a vaccine SPL document: \n{text}
"""

In [20]:
sections = rx_spl_df['contra_text'].tolist()[:29]
msgs = build_messages_from_iter(system, user, sections)

In [11]:
import torch

# Enable TF32 for faster matrix multiplication
torch.set_float32_matmul_precision('high')

In [21]:
out = [model.generate(m, max_new_tokens=512) for m in msgs]

skipping cudagraphs due to skipping cudagraphs due to multiple devices: device(type='cuda', index=0), device(type='cuda', index=1), device(type='cuda', index=2), device(type='cuda', index=3)


In [15]:
import json
from typing import Any, Optional, Tuple

def _find_json_span(text: str) -> Optional[Tuple[int, int]]:
    """
    Heuristically find the first top-level {...} or [...] span in the text
    using a simple stack-based scanner. Returns (start, end) indices or None.
    """
    for open_ch, close_ch in [("{", "}"), ("[", "]")]:
        start = text.find(open_ch)
        while start != -1:
            depth = 0
            for i, ch in enumerate(text[start:], start):
                if ch == open_ch:
                    depth += 1
                elif ch == close_ch:
                    depth -= 1
                    if depth == 0:
                        return start, i + 1  # end index is exclusive
            # If we got here, parentheses didn't balance; try another starting point
            start = text.find(open_ch, start + 1)
    return None


def extract_json(text: str) -> Optional[Any]:
    """
    Extract and parse a JSON object or array from a model's output.
    Returns Python data (dict/list) or None if parsing fails.
    """
    text = text.strip()

    # 1. Try direct JSON parse first
    try:
        return json.loads(text)
    except Exception:
        pass

    # 2. Try to locate a {...} or [...] span
    span = _find_json_span(text)
    if not span:
        return None

    start, end = span
    json_like = text[start:end].strip()

    # 3. Try parsing the extracted slice
    try:
        return json.loads(json_like)
    except Exception:
        pass

    # 4. Last-resort mild "repair": common minor issues
    repaired = (
        json_like
        .replace(",]", "]")
        .replace(",}", "}")
    )
    try:
        return json.loads(repaired)
    except Exception:
        return None


In [40]:
extract_json(out[4])

{'items': [{'span_text': 'should not be given to persons (a) whose immunologic responses are impaired because of HIV infections, congenital immunodeficiency such as chronic granulomatous disease or interferon gamma receptor deficiency, leukemia, lymphoma, or generalized malignancy or (b) whose immunologic responses have been suppressed by steroids, alkylating agents, antimetabolites, or radiation',
   'condition_text': 'impaired immunologic responses'},
  {'span_text': 'BCG VACCINE should not be administered to HIV-infected or immunocompromised infants, children, or adults',
   'condition_text': 'HIV-infected or immunocompromised'},
  {'span_text': 'Allergy to any component of BCG VACCINE or an anaphylactic or allergic reaction to a previous dose of BCG VACCINE are contraindications for vaccination',
   'condition_text': 'allergy to BCG vaccine component or previous reaction'},
  {'span_text': 'BCG VACCINE should not be used in infants, children, or adults with severe immune deficiency

In [ ]:
json_list = []
for i in out:
    res = extract_json(i)
    json_list.append(res)

In [ ]:
condition_texts = []
for j in json_list:
    ind_list = []
    if j and 'items' in j:
        for item in j['items']:
            ind_list.append(item['condition_text'])
    else:
        ind_list.append(None)
    condition_texts.append(ind_list)

In [39]:
cd = pd.DataFrame({'condition_texts': condition_texts})
cd

,condition_texts
0,"[pregnancy, severe allergic reaction to Adenov..."
1,[history of anaphylactic or anaphylactic-like ...
2,[history of severe allergic reaction to CYFEND...
3,[]
4,"[impaired immunologic responses, HIV-infected ..."
5,[]
6,[history of severe allergic reaction to vaccin...
7,[history of severe allergic reaction to VAXCHO...
8,[severe allergic reaction to vaccine component]
9,[severe allergic reaction to vaccine component]
